Environment Setup and Data Loading

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pickle
from google.colab import drive

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"hardware selected -> {device}")
if device.type == 'cpu':
  print("warning!! Cpu is in use, switch to gpu!")

print("\nMounting drive..")
drive.mount('/content/drive')
base_path = '/content/drive/My Drive/resources/KuaiRec 2.0/data/processed/'

print("\nLoading the processed data....")
with open(base_path + 'train_sequence.pkl', 'rb') as f:
  train_sequences = pickle.load(f)

with open(base_path + 'ipw_weights.pkl', 'rb') as f:
  ipw_weights = pickle.load(f)

with open(base_path + 'meta.pkl', 'rb') as f:
    meta = pickle.load(f)

print("data loaded")
print(f"number of users: {len(train_sequences)}")
print(f"number of weighted items: {len(ipw_weights)}")

hardware selected -> cuda

Mounting drive..
Mounted at /content/drive

Loading the processed data....
data loaded
number of users: 7176
number of weighted items: 10706


Metadata and Tensor prep

In [ ]:
num_users = meta['num_users']
num_items = meta['num_items']
num_categories = meta['num_cats']

#sample view
print(f"max user_id: {num_users}")
print(f"max item_id: {num_items}")
print(f"max category_id: {num_categories}")

#weight tensors
weight_tensor = torch.ones(num_items).to(device)
for item_id, weight in ipw_weights.items():
  if item_id < num_items:
    weight_tensor[item_id] = weight
print("weight tensors are created")

max user_id: 7176
max item_id: 10719
max category_id: 31
weight tensors are created


Pairs and Negative Sampling

In [ ]:
class TwoTowerDataset(Dataset):
  def __init__(self, sequences, num_items, category_map):
    self.sequences = sequences
    self.num_items = num_items
    self.samples = []

    for uid, data in sequences.items():
      items = data['item_sequence']
      cats = data['category_sequence']

      user_history = set(items)

      for i, c in zip(items, cats):
        self.samples.append((uid, i, c, user_history))

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):
    #positive sampling
    user_id, pos_item, pos_cat, history = self.samples[idx]

    #negative sampling: something that user hasn't seen, random item
    while True:
      neg_item = np.random.randint(1, self.num_items)
      if neg_item not in history:
        break

    #negative items category set to 0 for model to rely on item_id
    neg_cat = 0

    return {
        'user': torch.tensor(user_id, dtype = torch.long),
        'pos_item': torch.tensor(pos_item, dtype=torch.long),
        'pos_cat': torch.tensor(pos_cat, dtype=torch.long),
        'neg_item': torch.tensor(neg_item, dtype=torch.long),
        'neg_cat': torch.tensor(neg_cat, dtype=torch.long)
    }

dataset = TwoTowerDataset(train_sequences, num_items, category_map =None)
dataloader = DataLoader(dataset, batch_size = 1024, shuffle = True)

print(f"dataset initialized. total training samples: {len(dataset)}")
sample = next(iter(dataloader))
print("\nsample batch keys")
print(sample.keys())

dataset initialized. total training samples: 4231052

sample batch keys
dict_keys(['user', 'pos_item', 'pos_cat', 'neg_item', 'neg_cat'])


Model Architecture

In [ ]:
class CloudTwoTower(nn.Module):
  def __init__(self, num_users, num_items, num_cats, embed_dim = 64):
    super(CloudTwoTower, self).__init__()

    #user tower
    self.user_embedding = nn.Embedding(num_users, embed_dim)

    #item tower
    self.item_embedding = nn.Embedding(num_items, embed_dim)
    self.cat_embedding = nn.Embedding(num_cats, embed_dim)

    #initialize
    nn.init.normal_(self.user_embedding.weight, std = 0.01)
    nn.init.normal_(self.item_embedding.weight, std = 0.01)
    nn.init.normal_(self.cat_embedding.weight, std = 0.01)

  def forward(self, user_id, item_id, cat_id):
    u_vec = self.user_embedding(user_id)
    i_vec = self.item_embedding(item_id)
    c_vec = self.cat_embedding(cat_id)

    #element-wise sum
    final_item_vec = i_vec + c_vec

    #user vector * item vector
    dot = (u_vec * final_item_vec).sum(dim = 1)

    output = torch.sigmoid(dot)

    return output, final_item_vec

Training Loop

In [ ]:
model = CloudTwoTower(num_users, num_items, num_categories).to(device)
optimizer = optim.Adam(model.parameters(), lr = 0.001)

epochs = 5
loss_func = nn.BCELoss(reduction = 'none')

for epoch in range(epochs):
  total_loss = 0
  steps = 0

  model.train()

  for batch in dataloader:
    user = batch['user'].to(device)
    pos_item = batch['pos_item'].to(device)
    pos_cat = batch['pos_cat'].to(device)
    neg_item = batch['neg_item'].to(device)
    neg_cat = batch['neg_cat'].to(device)

    optimizer.zero_grad()

    #positive samples
    pos_pred, _ = model(user, pos_item, pos_cat)
    pos_label = torch.ones_like(pos_pred)

    loss_pos = loss_func(pos_pred, pos_label)

    weights = weight_tensor[pos_item]
    weighted_loss_pos = (loss_pos * weights).mean()

    #negative samples
    neg_pred, _ = model(user, neg_item, neg_cat)
    neg_label = torch.zeros_like(neg_pred)

    loss_neg = loss_func(neg_pred, neg_label).mean()
    total_batch_loss = weighted_loss_pos + loss_neg

    total_batch_loss.backward()

    optimizer.step()

    total_loss += total_batch_loss.item()
    steps += 1

  print(f"Epoch {epoch+1}/{epochs}, Avg Loss: {total_loss/steps:.4f}")

torch.save(model.state_dict(), base_path + 'checkpoint_model.pth')
print("training completed")
print("checkpoint saved")

Epoch 1/5, Avg Loss: 0.0340
Epoch 2/5, Avg Loss: 0.0007
Epoch 3/5, Avg Loss: 0.0006
Epoch 4/5, Avg Loss: 0.0006
Epoch 5/5, Avg Loss: 0.0005
training completed
checkpoint saved


Freezing the Embedding

In [ ]:
print("extracting frozen embedding..")

model.eval()

item_to_cat = {}
for uid, data in train_sequences.items():
  items = data['item_sequence']
  cats = data['category_sequence']
  for i, c in zip(items, cats):
    item_to_cat[i] = c

frozen_matrix = np.zeros((num_items, 64))

with torch.no_grad():
  for i in range(num_items):
    c = item_to_cat.get(i, 0)

    i_tensor = torch.tensor([i], dtype = torch.long).to(device)
    c_tensor = torch.tensor([c], dtype = torch.long).to(device)

    #only item tower components
    i_emb = model.item_embedding(i_tensor)
    c_emb = model.cat_embedding(c_tensor)

    final_vec = i_emb + c_emb

    frozen_matrix[i] = final_vec.cpu().numpy()

np.save(base_path + 'frozen_item_embeddings.npy', frozen_matrix)

print(f"extraction completed")
print(f"frozen matrix shape: {frozen_matrix.shape}")
print(f"saved to: {base_path}frozen_item_embeddings.npy")

extracting frozen embedding..
extraction completed
frozen matrix shape: (10719, 64)
saved to: /content/drive/My Drive/resources/KuaiRec 2.0/data/processed/frozen_item_embeddings.npy
